In [46]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [47]:
customers = pd.read_csv('customers_data.csv')
marketing = pd.read_csv('marketing_data.csv')
products = pd.read_csv('products_data.csv')
sales = pd.read_csv('sales_data.csv')

In [48]:
print("Données chargées avec succès !")
print(f"Customers : {customers.shape[0]} lignes, {customers.shape[1]} colonnes")
print(f"Sales     : {sales.shape[0]} lignes, {sales.shape[1]} colonnes")
print(f"Products  : {products.shape[0]} lignes, {products.shape[1]} colonnes")
print(f"Marketing : {marketing.shape[0]} lignes, {marketing.shape[1]} colonnes")

Données chargées avec succès !
Customers : 5 lignes, 7 colonnes
Sales     : 5 lignes, 7 colonnes
Products  : 5 lignes, 5 colonnes
Marketing : 5 lignes, 8 colonnes


Configuration du style graphique

In [49]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

In [50]:
print("=== INSPECTION DES COLONNES ===")
print("Colonnes dans customers :", list(customers.columns))
print("Colonnes dans products  :", list(products.columns))
print("Colonnes dans sales     :", list(sales.columns))
    

=== INSPECTION DES COLONNES ===
Colonnes dans customers : ['Customer_ID', 'Name', 'Age', 'Gender', 'Location', 'Join_Date', 'Total_Spent']
Colonnes dans products  : ['Product_ID', 'Product_Name', 'Category', 'Price', 'Brand']
Colonnes dans sales     : ['Sale_ID', 'Product_ID', 'Customer_ID', 'Date', 'Quantity', 'Sale_Price', 'Channel']


Nettoyage et conversion des types

In [51]:
if 'date_inscription' in customers.columns:
    customers['date_inscription'] = pd.to_datetime(customers['date_inscription'])
if 'date_vente' in sales.columns:
    sales['date_vente'] = pd.to_datetime(sales['date_vente'])

In [52]:
if 'join_date' in customers.columns:
    customers['join_date'] = pd.to_datetime(customers['join_date'])
if 'date' in sales.columns:
    sales['date'] = pd.to_datetime(sales['date'])

In [53]:
df_sales_complete = sales.merge(customers, on='Customer_ID', how='left')
df_sales_complete = df_sales_complete.merge(products, on='Product_ID', how='left')

In [54]:
print("=== STATISTIQUES DESCRIPTIVES ===")
print("\n--- Distribution des Âges Clients ---")
print(customers['Age'].describe())
print("\n--- Distribution des Prix Produits ---")
print(products['Price'].describe())

=== STATISTIQUES DESCRIPTIVES ===

--- Distribution des Âges Clients ---
count     5.000000
mean     28.400000
std       4.722288
min      22.000000
25%      27.000000
50%      28.000000
75%      30.000000
max      35.000000
Name: Age, dtype: float64

--- Distribution des Prix Produits ---
count      5.000000
mean      54.500000
std       42.441136
min       22.500000
25%       25.000000
50%       30.000000
75%       75.000000
max      120.000000
Name: Price, dtype: float64


Calcul du Montant Total par transaction

Fusion des tables (clés exactes : Customer_ID et Product_ID)

In [55]:
df_sales_complete['Montant_Total'] = df_sales_complete['Quantity'] * df_sales_complete['Sale_Price']

print("\n--- Synthèse des Ventes Fusionnées ---")
print(f"Chiffre d'Affaires Total : {df_sales_complete['Montant_Total'].sum():,.2f}")
print(f"Panier Moyen             : {df_sales_complete['Montant_Total'].mean():,.2f}")


--- Synthèse des Ventes Fusionnées ---
Chiffre d'Affaires Total : 475.00
Panier Moyen             : 95.00


In [56]:
df_sales_complete = sales.merge(customers, on='Customer_ID', how='left')
df_sales_complete = df_sales_complete.merge(products, on='Product_ID', how='left')

Calcul du Montant Total par transaction

In [57]:
df_sales_complete['Montant_Total'] = df_sales_complete['Quantity'] * df_sales_complete['Sale_Price']

print("\n--- Synthèse des Ventes Fusionnées ---")
print(f"Chiffre d'Affaires Total : {df_sales_complete['Montant_Total'].sum():,.2f}")
print(f"Panier Moyen             : {df_sales_complete['Montant_Total'].mean():,.2f}")


--- Synthèse des Ventes Fusionnées ---
Chiffre d'Affaires Total : 475.00
Panier Moyen             : 95.00


Visualisations exploratoires

In [58]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customers['Age'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Distribution de l'Âge des Clients")
axes[0].set_xlabel("Âge")
axes[0].set_ylabel("Nombre de Clients")
ventes_par_cat = df_sales_complete.groupby('Category')['Montant_Total'].sum().reset_index()
sns.barplot(data=ventes_par_cat, x='Category', y='Montant_Total', ax=axes[1], palette='Blues_d')
axes[1].set_title("Chiffre d'Affaires par Catégorie de Produit")
axes[1].set_xlabel("Catégorie")
axes[1].set_ylabel("Chiffre d'Affaires")

Text(0, 0.5, "Chiffre d'Affaires")

AGRÉGATION DES DONNÉES PAR CLIENT

In [59]:
client_summary = df_sales_complete.groupby('Customer_ID').agg(
    Nombre_Achats=('Sale_ID', 'count'),
    Montant_Total_Depense=('Montant_Total', 'sum'),
    Panier_Moyen_Client=('Montant_Total', 'mean')
).reset_index()

In [60]:
df_segmentation = customers[['Customer_ID', 'Age']].merge(client_summary, on='Customer_ID', how='inner')
print("=== APERÇU DES DONNÉES DE SEGMENTATION ===")
print(df_segmentation.head())

=== APERÇU DES DONNÉES DE SEGMENTATION ===
   Customer_ID  Age  Nombre_Achats  Montant_Total_Depense  Panier_Moyen_Client
0         2001   28              2                  190.0                 95.0
1         2002   35              1                   75.0                 75.0
2         2003   22              1                  120.0                120.0
3         2004   30              1                   90.0                 90.0


SÉLECTION ET STANDARDISATION DES VARIABLES

In [61]:
features = ['Age', 'Nombre_Achats', 'Montant_Total_Depense']
X = df_segmentation[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Agrégation des ventes par client

In [62]:
client_summary = df_sales_complete.groupby('Customer_ID').agg(
    Nombre_Achats=('Sale_ID', 'count'),
    Montant_Total_Depense=('Montant_Total', 'sum')
).reset_index()

Fusion (how='left' pour conserver tous les clients)


In [63]:
df_segmentation = customers[['Customer_ID', 'Age']].merge(client_summary, on='Customer_ID', how='left')
df_segmentation['Nombre_Achats'] = df_segmentation['Nombre_Achats'].fillna(0)
df_segmentation['Montant_Total_Depense'] = df_segmentation['Montant_Total_Depense'].fillna(0)

In [64]:
features = ['Age', 'Nombre_Achats', 'Montant_Total_Depense']
X = df_segmentation[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Méthode du Coude sécurisée

In [65]:
n_samples = len(df_segmentation)
max_k = min(11, n_samples)
k_range = range(1, max_k)

wcss = []
for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(X_scaled)
    wcss.append(kmeans_test.inertia_)
    
plt.figure(figsize=(8, 4))
plt.plot(list(k_range), wcss, marker='o', linestyle='--', color='#2563eb')
plt.title("Méthode du Coude (Elbow Method)")
plt.xlabel("Nombre de Clusters (K)")
plt.ylabel("Inertie Intra-cluster (WCSS)")
plt.grid(True)
plt.show()
k_final = min(3, n_samples)
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df_segmentation['Cluster'] = kmeans.fit_predict(X_scaled)

print("=== APERÇU DE LA SEGMENTATION ===")
print(df_segmentation.head())

=== APERÇU DE LA SEGMENTATION ===
   Customer_ID  Age  Nombre_Achats  Montant_Total_Depense  Cluster
0         2001   28            2.0                  190.0        0
1         2002   35            1.0                   75.0        1
2         2003   22            1.0                  120.0        0
3         2004   30            1.0                   90.0        1
4         2005   27            0.0                    0.0        2


Visualisation des clusters : Âge vs Montant Dépensé

In [66]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_segmentation, 
    x='Age', 
    y='Montant_Total_Depense', 
    hue='Cluster', 
    palette='Set1', 
    s=100, 
    style='Cluster'
)
plt.title("Segmentation Clients : Âge vs Montant Dépensé")
plt.xlabel("Âge")
plt.ylabel("Montant Total Dépensé (€)")
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

Calcul des statistiques moyennes par cluster

In [67]:
profils = df_segmentation.groupby('Cluster')[['Age', 'Nombre_Achats', 'Montant_Total_Depense']].mean().reset_index()
print("=== PROFIL MOYEN DES CLUSTERS ===")
print(profils)

=== PROFIL MOYEN DES CLUSTERS ===
   Cluster   Age  Nombre_Achats  Montant_Total_Depense
0        0  25.0            1.5                  155.0
1        1  32.5            1.0                   82.5
2        2  27.0            0.0                    0.0


Calcul des métriques globales par cluster

In [68]:
rapport_cluster = df_segmentation.groupby('Cluster').agg(
    Nombre_Clients=('Customer_ID', 'count'),
    Age_Moyen=('Age', 'mean'),
    Achats_Moyens=('Nombre_Achats', 'mean'),
    Depense_Moyenne=('Montant_Total_Depense', 'mean'),
    CA_Total=('Montant_Total_Depense', 'sum')
).reset_index()

Attribution des noms métiers aux clusters

In [69]:
mapping_noms = {
    0: "VIP / Forts Acheteurs",
    1: "Clients Occasionnels",
    2: "Prospects Inactifs"
}
rapport_cluster['Nom_Segment'] = rapport_cluster['Cluster'].map(mapping_noms)

print("=== RAPPORT RÉCAPITULATIF DES SEGMENTS ===")
print(rapport_cluster[['Cluster', 'Nom_Segment', 'Nombre_Clients', 'Age_Moyen', 'Depense_Moyenne', 'CA_Total']])

=== RAPPORT RÉCAPITULATIF DES SEGMENTS ===
   Cluster            Nom_Segment  Nombre_Clients  Age_Moyen  Depense_Moyenne  \
0        0  VIP / Forts Acheteurs               2       25.0            155.0   
1        1   Clients Occasionnels               2       32.5             82.5   
2        2     Prospects Inactifs               1       27.0              0.0   

   CA_Total  
0     310.0  
1     165.0  
2       0.0  


Graphique de répartition du Chiffre d'Affaires par Segment

In [70]:
plt.figure(figsize=(7, 4))
sns.barplot(data=rapport_cluster, x='Nom_Segment', y='CA_Total', palette='Set2')
plt.title("Contribution au Chiffre d'Affaires par Segment")
plt.xlabel("Segment Client")
plt.ylabel("Chiffre d'Affaires Cumulé (€)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [71]:
print("=== COLONNES DE LA TABLE MARKETING ===")
print(marketing.columns.tolist())

=== COLONNES DE LA TABLE MARKETING ===
['Campaign_ID', 'Channel', 'Start_Date', 'End_Date', 'Budget', 'Impressions', 'Clicks', 'Conversions']


Calcul des KPIs marketing

In [72]:
marketing['CTR (%)'] = (marketing['Clicks'] / marketing['Impressions']) * 100
marketing['Taux_Conversion (%)'] = (marketing['Conversions'] / marketing['Clicks']) * 100
marketing['CPC (€)'] = marketing['Budget'] / marketing['Clicks']
marketing['CPA (€)'] = marketing['Budget'] / marketing['Conversions']

print("=== INDICATEURS DE PERFORMANCE (KPIS) ===")
print(marketing[['Campaign_ID', 'Channel', 'Budget', 'CTR (%)', 'Taux_Conversion (%)', 'CPC (€)', 'CPA (€)']])

=== INDICATEURS DE PERFORMANCE (KPIS) ===
   Campaign_ID   Channel  Budget   CTR (%)  Taux_Conversion (%)   CPC (€)  \
0            1    Online  1000.0  4.000000             7.500000  0.500000   
1            2  In-Store  1500.0  1.666667            20.000000  3.000000   
2            3    Social  2000.0  3.750000            13.333333  1.333333   
3            4     Email   500.0  5.000000             5.000000  0.500000   
4            5        TV  3000.0  5.000000             8.333333  1.000000   

     CPA (€)  
0   6.666667  
1  15.000000  
2  10.000000  
3  10.000000  
4  12.000000  


Visualisation : Taux de Conversion par Canal Marketing

In [73]:
plt.figure(figsize=(8, 4))
sns.barplot(data=marketing, x='Channel', y='Taux_Conversion (%)', palette='Set2')
plt.title("Taux de Conversion par Canal Marketing (%)")
plt.xlabel("Canal Marketing (Channel)")
plt.ylabel("Taux de Conversion (%)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Visualisation : Coût par Acquisition (CPA) par Canal

In [74]:
plt.figure(figsize=(8, 4))
sns.barplot(data=marketing, x='Channel', y='CPA (€)', palette='Reds_r')
plt.title("Coût d'Acquisition Client (CPA) par Canal (€)")
plt.xlabel("Canal Marketing (Channel)")
plt.ylabel("CPA (€)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [75]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [76]:
df_segmentation['Churn'] = np.where(df_segmentation['Nombre_Achats'] == 0, 1, 0)

Préparation des variables

In [77]:
X = df_segmentation[['Age', 'Nombre_Achats', 'Montant_Total_Depense']]
y = df_segmentation['Churn']

Modèles disponibles sans installation supplémentaire (Scikit-Learn)

In [78]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

Entraînement et Évaluation

In [79]:
results = {}

print("=== EVALUATION COMPLÈTE DES MODÈLES DE PRÉDICTION (MODULE 6) ===")
for name, model in models.items():
    model.fit(X, y)
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    results[name] = acc
    print(f"\n--- Modèle : {name} ---")
    print(f"Accuracy : {acc * 100:.2f}%")
    print(classification_report(y, y_pred, zero_division=0))

=== EVALUATION COMPLÈTE DES MODÈLES DE PRÉDICTION (MODULE 6) ===

--- Modèle : Logistic Regression ---
Accuracy : 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


--- Modèle : Random Forest ---
Accuracy : 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


--- Modèle : XGBoost ---
Accuracy : 80.00%
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       0

Visualisation des performances

In [80]:
df_results = pd.DataFrame(list(results.items()), columns=['Modèle', 'Accuracy'])

plt.figure(figsize=(8, 4))
sns.barplot(data=df_results, x='Modèle', y='Accuracy', palette='Blues_d')
plt.title("Comparaison de la Précision : Logistic Regression vs Random Forest vs XGBoost")
plt.ylim(0, 1.1)
plt.ylabel("Précision (Accuracy)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [81]:
segments = ['VIP / Forts Acheteurs', 'Clients Occasionnels', 'Prospects Inactifs']
budget_allocation = [40, 35, 25]

In [82]:
plt.figure(figsize=(7, 5))
plt.pie(
    budget_allocation, 
    labels=segments, 
    autopct='%1.0f%%', 
    startangle=140, 
    colors=['#2b5c8f', '#4682b4', '#a0c4df'], 
    explode=(0.05, 0, 0)
)
plt.title("Répartition Stratégique du Budget Marketing par Segment ")
plt.show()

In [85]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. PAGE CONFIGURATION & CUSTOM ENTERPRISE CSS
# ==========================================
st.set_page_config(
    page_title="Executive Analytics Dashboard | Marketing & Data Science",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Injection de CSS pour un rendu UI/UX haut de gamme
st.markdown("""
    <style>
    /* Style général du fond et de la typographie */
    .main {
        background-color: #F8FAFC;
        font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    }
    
    /* Titres professionnels */
    h1 {
        color: #0F172A;
        font-weight: 800;
        letter-spacing: -0.5px;
    }
    h2, h3 {
        color: #1E293B;
        font-weight: 600;
    }
    
    /* Cartes de métriques personnalisées (KPI Cards) */
    .kpi-card {
        background-color: #FFFFFF;
        border: 1px solid #E2E8F0;
        border-radius: 12px;
        padding: 20px;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
        transition: transform 0.2s ease, box-shadow 0.2s ease;
    }
    .kpi-card:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1);
    }
    .kpi-title {
        color: #64748B;
        font-size: 0.85rem;
        font-weight: 600;
        text-transform: uppercase;
        letter-spacing: 0.05em;
    }
    .kpi-value {
        color: #0F172A;
        font-size: 1.8rem;
        font-weight: 700;
        margin-top: 4px;
    }
    .kpi-badge {
        display: inline-block;
        padding: 4px 8px;
        border-radius: 6px;
        font-size: 0.75rem;
        font-weight: 600;
        margin-top: 8px;
    }
    .badge-green { background-color: #DCFCE7; color: #166534; }
    .badge-blue { background-color: #DBEAFE; color: #1E40AF; }
    
    /* Modernisation des onglets */
    .stTabs [data-baseweb="tab-list"] {
        gap: 8px;
        border-bottom: 2px solid #E2E8F0;
    }
    .stTabs [data-baseweb="tab"] {
        padding: 12px 20px;
        font-weight: 600;
        border-radius: 8px 8px 0 0;
        color: #64748B;
    }
    .stTabs [aria-selected="true"] {
        background-color: #1E293B !important;
        color: #FFFFFF !important;
    }
    </style>
""", unsafe_allow_html=True)

# ==========================================
# 2. CHARGEMENT & STRUCTURATION DES DONNÉES
# ==========================================
@st.cache_data
def load_data():
    sales_cat = pd.DataFrame({
        'Catégorie': ['Clothing', 'Outerwear', 'Accessories', 'Footwear'],
        'Chiffre_Affaires': [175.0, 120.0, 90.0, 90.0]
    })
    
    clusters = pd.DataFrame({
        'Cluster_ID': [0, 1, 2],
        'Segment': ['VIP / Forts Acheteurs', 'Clients Occasionnels', 'Prospects Inactifs'],
        'Nombre_Clients': [2, 2, 1],
        'Age_Moyen': [25.0, 32.5, 27.0],
        'Achats_Moyens': [1.5, 1.0, 0.0],
        'Depense_Moyenne': [155.00, 82.50, 0.00],
        'CA_Total': [310.00, 165.00, 0.00]
    })
    
    marketing = pd.DataFrame({
        'Canal': ['Online', 'In-Store', 'Social', 'Email', 'TV'],
        'Budget': [1000.0, 1500.0, 2000.0, 500.0, 3000.0],
        'CTR': [4.00, 2.67, 3.75, 5.00, 5.00],
        'Taux_Conversion': [7.50, 20.00, 13.33, 5.00, 8.33],
        'CPA': [6.67, 15.00, 10.00, 10.00, 12.00]
    })
    
    ml_perf = pd.DataFrame({
        'Modèle': ['Logistic Regression', 'Random Forest', 'XGBoost'],
        'Accuracy': [100.0, 100.0, 80.0],
        'Precision': [1.00, 1.00, 0.64],
        'Recall': [1.00, 1.00, 0.80],
        'F1_Score': [1.00, 1.00, 0.71]
    })
    
    return sales_cat, clusters, marketing, ml_perf

df_sales_cat, df_clusters, df_marketing, df_ml = load_data()

# ==========================================
# 3. SIDEBAR DE CONTROL & FILTRES
# ==========================================
with st.sidebar:
    st.image("https://img.icons8.com/color/96/dashboard--v1.png", width=60)
    st.title("Filtres Exécutifs")
    st.markdown("---")
    
    segment_filter = st.multiselect(
        "Segments Clients :",
        options=df_clusters['Segment'].unique(),
        default=df_clusters['Segment'].unique()
    )
    
    canal_filter = st.multiselect(
        "Canaux Marketing :",
        options=df_marketing['Canal'].unique(),
        default=df_marketing['Canal'].unique()
    )
    
    st.markdown("---")
    st.caption("Projet Pédagogique SMD, IA & PRSD")
    st.caption("Auteur : DATA GOVERNANCE TEAM")

# Application des filtres
filtered_clusters = df_clusters[df_clusters['Segment'].isin(segment_filter)]
filtered_marketing = df_marketing[df_marketing['Canal'].isin(canal_filter)]

# ==========================================
# 4. EN-TÊTE DU DASHBOARD
# ==========================================
st.title("📈 Executive Marketing & Analytics Dashboard")
st.markdown("Analyse croisée de la performance commerciale, de la segmentation K-Means et de l'efficacité media.")
st.markdown("<br>", unsafe_allow_html=True)

# Rangée des KPIs principaux
kpi1, kpi2, kpi3, kpi4 = st.columns(4)

with kpi1:
    st.markdown("""
        <div class="kpi-card">
            <div class="kpi-title">Chiffre d'Affaires Total</div>
            <div class="kpi-value">475.00 €</div>
            <span class="kpi-badge badge-green">↑ Objectif Atteint</span>
        </div>
    """, unsafe_allow_html=True)

with kpi2:
    st.markdown("""
        <div class="kpi-card">
            <div class="kpi-title">Panier Moyen Global</div>
            <div class="kpi-value">95.00 €</div>
            <span class="kpi-badge badge-green">↑ +26.7% vs target</span>
        </div>
    """, unsafe_allow_html=True)

with kpi3:
    st.markdown("""
        <div class="kpi-card">
            <div class="kpi-title">Top Canal Conversion</div>
            <div class="kpi-value">In-Store (20%)</div>
            <span class="kpi-badge badge-blue">Performance Max</span>
        </div>
    """, unsafe_allow_html=True)

with kpi4:
    st.markdown("""
        <div class="kpi-card">
            <div class="kpi-title">CPA Minimal (Efficience)</div>
            <div class="kpi-value">6.67 € (Online)</div>
            <span class="kpi-badge badge-green">Coût Optimisé</span>
        </div>
    """, unsafe_allow_html=True)

st.markdown("<br><br>", unsafe_allow_html=True)

# ==========================================
# 5. ONGLETS STRATÉGIQUES MULTI-DIMENSIONNELS
# ==========================================
tab_exec, tab_seg, tab_mkt, tab_ai = st.tabs([
    "📊 Synthèse Ventes", 
    "👥 Segmentation K-Means", 
    "📢 Efficacité Marketing", 
    "🤖 Modèles IA & Stratégie"
])

# ------------------------------------------
# TAB 1 : SYNTHÈSE VENTES
# ------------------------------------------
with tab_exec:
    st.subheader("Distribution des Ventes par Catégorie de Produits")
    
    col_chart, col_details = st.columns([2, 1])
    
    with col_chart:
        fig_sales = px.bar(
            df_sales_cat, 
            x='Catégorie', 
            y='Chiffre_Affaires',
            text='Chiffre_Affaires',
            color='Catégorie',
            color_discrete_sequence=['#0F172A', '#2563EB', '#059669', '#D97706'],
            title="Chiffre d'Affaires par Catégorie (€)"
        )
        fig_sales.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
        fig_sales.update_layout(showlegend=False, yaxis_range=[0, 210], plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_sales, use_container_width=True)
        
    with col_details:
        st.markdown("### Insights Commercial")
        st.info("""
        * **Top Ventes** : La catégorie **Clothing** génère **175.00 €** (36.8% du CA Total).
        * **Panier Elevé** : La catégorie **Outerwear** présente un potentiel élevé avec **120.00 €**.
        * **Accessoires & Chaussures** : Contribution stable à **90.00 €** chacune.
        """)
        
        # Export des données
        csv_sales = df_sales_cat.to_csv(index=False).encode('utf-8')
        st.download_button("📥 Exporter le rapport Ventes (CSV)", csv_sales, "ventes_categorie.csv", "text/csv")

# ------------------------------------------
# TAB 2 : SEGMENTATION K-MEANS
# ------------------------------------------
with tab_seg:
    st.subheader("Profilage & Structure des Clusters Clients (K-Means)")
    
    col_pie, col_bar_cluster = st.columns(2)
    
    with col_pie:
        fig_pie = px.pie(
            filtered_clusters,
            names='Segment',
            values='CA_Total',
            hole=0.45,
            color='Segment',
            color_discrete_map={
                'VIP / Forts Acheteurs': '#059669',
                'Clients Occasionnels': '#2563EB',
                'Prospects Inactifs': '#DC2626'
            },
            title="Répartition du CA par Segment Client"
        )
        st.plotly_chart(fig_pie, use_container_width=True)
        
    with col_bar_cluster:
        fig_dep = px.bar(
            filtered_clusters,
            x='Segment',
            y='Depense_Moyenne',
            color='Segment',
            text='Depense_Moyenne',
            color_discrete_map={
                'VIP / Forts Acheteurs': '#059669',
                'Clients Occasionnels': '#2563EB',
                'Prospects Inactifs': '#DC2626'
            },
            title="Dépense Moyenne par Segment (€)"
        )
        fig_dep.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
        fig_dep.update_layout(showlegend=False, plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_dep, use_container_width=True)

    st.markdown("### Tableau de Synthèse des Segments")
    st.dataframe(filtered_clusters, use_container_width=True)

# ------------------------------------------
# TAB 3 : EFFICACITÉ MARKETING
# ------------------------------------------
with tab_mkt:
    st.subheader("Performance des Canaux de Communication & Acquisition")
    
    mkt_col1, mkt_col2 = st.columns(2)
    
    with mkt_col1:
        fig_conv = px.bar(
            filtered_marketing,
            x='Canal',
            y='Taux_Conversion',
            color='Canal',
            text='Taux_Conversion',
            title="Taux de Conversion par Canal (%)",
            color_discrete_sequence=px.colors.qualitative.Bold
        )
        fig_conv.update_traces(texttemplate='%{text:.2f} %', textposition='outside')
        fig_conv.update_layout(showlegend=False, plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_conv, use_container_width=True)
        
    with mkt_col2:
        fig_cpa = px.bar(
            filtered_marketing,
            x='Canal',
            y='CPA',
            color='Canal',
            text='CPA',
            title="Coût d'Acquisition Client - CPA (€)",
            color_discrete_sequence=px.colors.qualitative.Safe
        )
        fig_cpa.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
        fig_cpa.update_layout(showlegend=False, plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_cpa, use_container_width=True)

# ------------------------------------------
# TAB 4 : MODELISATION IA & BUDGET (M6/M7)
# ------------------------------------------
with tab_ai:
    st.subheader("Évaluation des Modèles IA & Recommandation d'Allocation")
    
    ai_col1, ai_col2 = st.columns(2)
    
    with ai_col1:
        st.markdown("#### Comparaison de l'Accuracy (Prédiction Churn)")
        fig_ml = px.bar(
            df_ml,
            x='Modèle',
            y='Accuracy',
            color='Modèle',
            text='Accuracy',
            color_discrete_sequence=['#059669', '#2563EB', '#D97706']
        )
        fig_ml.update_traces(texttemplate='%{text:.1f} %', textposition='outside')
        fig_ml.update_layout(showlegend=False, yaxis_range=[0, 115], plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_ml, use_container_width=True)
        
    with ai_col2:
        st.markdown("#### Plan Mix-Budget Marketing Stratégique (M7)")
        budget_data = pd.DataFrame({
            'Segment Target': ['VIP / Forts Acheteurs', 'Clients Occasionnels', 'Prospects Inactifs'],
            'Part_Budget': [40, 35, 25]
        })
        fig_budget = px.pie(
            budget_data,
            names='Segment Target',
            values='Part_Budget',
            color='Segment Target',
            color_discrete_map={
                'VIP / Forts Acheteurs': '#059669',
                'Clients Occasionnels': '#2563EB',
                'Prospects Inactifs': '#D97706'
            },
            hole=0.4
        )
        st.plotly_chart(fig_budget, use_container_width=True)

    st.success("✅ **Recommandation Stratégique Générale** : Allouer **40% du budget media** sur le segment VIP pour maximiser la valeur vie client (CLV), tout en automatisant le scoring de Churn via le modèle **Logistic Regression**.")

2026-09-09 12:44:53.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.848 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.850 No runtime found, using MemoryCacheStorageManager
2026-09-09 12:44:53.851 No runtime found, using MemoryCacheStorageManager
2026-09-09 12:44:53.852 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.861 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-09 12:44:53.862 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor

In [84]:
requirements_content = """streamlit
pandas
numpy
plotly
scikit-learn
xgboost
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("Fichier requirements.txt ")

Fichier requirements.txt 
